# Week 2 – Logistics Data Collection, Cleaning & Preprocessing

This notebook demonstrates a practical preprocessing pipeline using the DataCo Smart Supply Chain for Big Data reference dataset.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

## 2. Load and Inspect Dataset

In [ ]:
df = pd.read_csv('../data/raw/DataCoSupplyChainDataset.csv')
print('Shape:', df.shape)
display(df.head())
df.info()

## 3. Missing Values

In [ ]:
missing = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_percentage': df.isna().mean()*100})
display(missing.sort_values('missing_percentage', ascending=False).head(20))

## 4. Standardize Columns and Remove Duplicates

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_', regex=False)
df = df.drop_duplicates().copy()
print('Shape after duplicate removal:', df.shape)

## 5. Correct Data Types

In [ ]:
for col in ['order_date_(dateorders)', 'shipping_date_(dateorders)']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

for col in ['sales', 'shipping_cost', 'order_item_quantity']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

## 6. Handle Missing Values

In [ ]:
for col in ['sales', 'shipping_cost']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

if 'shipping_mode' in df.columns and not df['shipping_mode'].mode().empty:
    df['shipping_mode'] = df['shipping_mode'].fillna(df['shipping_mode'].mode()[0])

## 7. Detect Outliers Using IQR

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

for col in ['sales', 'shipping_cost']:
    if col in df.columns:
        low, high = iqr_bounds(df[col].dropna())
        print(col, ((df[col] < low) | (df[col] > high)).sum(), 'outliers')

## 8. Standardize Numerical Features

In [ ]:
features = [c for c in ['sales', 'shipping_cost', 'order_item_quantity'] if c in df.columns]
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

## 9. Final Validation and Export

In [ ]:
print('Final shape:', df.shape)
print('Duplicates:', df.duplicated().sum())
print('Missing values:', df.isna().sum().sum())
df.to_csv('../data/processed/cleaned_logistics_data.csv', index=False)
print('Saved cleaned dataset.')

## 10. Conclusion

The preprocessing workflow improves consistency, handles missing data, evaluates outliers, and scales numerical variables for downstream logistics analytics.